# Fine-Tune RT-DETR on Moku Dataset

Fine-tune [RT-DETR r18vd](https://huggingface.co/PekingU/rtdetr_r18vd) on the `kaya-go/moku-v1` dataset for goban stone detection (3 classes: `board`, `black_stone`, `white_stone`).

**Pipeline:**
1. Load dataset from HF Hub
2. Load pre-trained RT-DETR and configure for 3 classes
3. Train with HF `Trainer` (visible training loop)
4. Evaluate with COCO mAP metrics
5. Push best model to HF Hub

In [1]:
%load_ext autoreload
%autoreload 2

from datasets import load_dataset
from transformers import Trainer, TrainingArguments

from moku.dataset import CATEGORIES, ID_TO_CATEGORY
from moku.training import (
    HF_DATASET,
    collate_fn,
    evaluate_map,
    format_map_results,
    load_image_processor,
    load_model,
    make_eval_transform,
    make_train_transform,
)

## Load Dataset

In [2]:
dataset = load_dataset(HF_DATASET)
print(dataset)
print(f"\nCategories: {CATEGORIES}")
print(f"Labels: {ID_TO_CATEGORY}")

DatasetDict({
    train: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 382
    })
    validation: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 53
    })
    test: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 50
    })
})

Categories: {'black_stone': 0, 'white_stone': 1, 'board_corner': 2}
Labels: {0: 'black_stone', 1: 'white_stone', 2: 'board_corner'}


## Load Model & Image Processor

Load RT-DETR r18vd pre-trained on COCO. The classification head is re-initialized for our 3 categories (`ignore_mismatched_sizes=True`).

In [3]:
image_processor = load_image_processor()
model = load_model()

# Print trainable parameter count
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")

Loading weights:   0%|          | 0/526 [00:00<?, ?it/s]

RTDetrForObjectDetection LOAD REPORT from: PekingU/rtdetr_r18vd
Key                                        | Status   |                                                                                        
-------------------------------------------+----------+----------------------------------------------------------------------------------------
model.decoder.class_embed.{0, 1, 2}.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80]) vs model:torch.Size([3])          
model.enc_score_head.bias                  | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80]) vs model:torch.Size([3])          
model.denoising_class_embed.weight         | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([81, 256]) vs model:torch.Size([4, 256])
model.enc_score_head.weight                | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80, 256]) vs model:torch.Size([3, 256])
model.decoder.class_embed.{0, 1, 2}.weight | MISMATCH | Reinit due to si

Total parameters: 20,075,740
Trainable parameters: 20,075,740


## Prepare Dataset Transforms

Apply image processor transforms. Training includes random horizontal flip. Evaluation uses only resize + normalize.

In [4]:
dataset["train"].set_transform(make_train_transform(image_processor))
dataset["validation"].set_transform(make_eval_transform(image_processor))
dataset["test"].set_transform(make_eval_transform(image_processor))

# Verify a sample
sample = dataset["train"][0]
print(f"pixel_values shape: {sample['pixel_values'].shape}")
print(f"labels keys: {sample['labels'].keys()}")
print(f"num objects: {len(sample['labels']['class_labels'])}")

pixel_values shape: torch.Size([3, 640, 640])
labels keys: KeysView({'size': tensor([640, 640]), 'image_id': tensor([0]), 'class_labels': tensor([2, 2, 2, 2]), 'boxes': tensor([[0.2164, 0.2188, 0.0266, 0.0344],
        [0.7641, 0.2266, 0.0250, 0.0312],
        [0.1477, 0.9789, 0.0422, 0.0422],
        [0.8250, 0.9809, 0.0344, 0.0367]]), 'area': tensor([374., 320., 729., 517.]), 'iscrowd': tensor([0, 0, 0, 0]), 'orig_size': tensor([640, 640])})
num objects: 4


In [ ]:
RUN_NAME = "baseline"
OUTPUT_DIR = f"runs/{RUN_NAME}"

training_args = TrainingArguments(
    project="moku",
    output_dir=OUTPUT_DIR,
    num_train_epochs=50,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    weight_decay=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    remove_unused_columns=False,
    dataloader_num_workers=0,
    fp16=False,
    use_cpu=True,  # MPS not fully supported; use GPU on HF Jobs
    push_to_hub=False,
    report_to="trackio",
    trackio_space_id="kaya-go/moku-training",
    run_name=RUN_NAME,
)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"LR: {training_args.learning_rate}")
print(f"Batch size: {training_args.per_device_train_batch_size}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Output directory: runs/baseline
Epochs: 50
LR: 0.0001
Batch size: 4


## Train

The HF `Trainer` runs the training loop. The RT-DETR model computes the loss internally (Hungarian matching + focal classification + L1/GIoU box regression). Training and eval loss are logged each epoch.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
)

trainer.train()

## Evaluate on Test Set

Compute COCO mAP metrics on the held-out test split using the best checkpoint.

In [7]:
metrics = evaluate_map(
    model=trainer.model,
    dataset=dataset["test"],
    image_processor=image_processor,
    batch_size=8,
    threshold=0.3,
)

display(format_map_results(metrics))

/Users/hadim/Code/libs/moku/.pixi/envs/default/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


,metric,value
0,mAP,0.000060
1,mAP@50,0.000176
2,mAP@75,0.000000
3,mAR@100,0.007143


## Push to Hugging Face Hub

Push the fine-tuned model and image processor to `kaya-go/moku-v1`.

In [ ]:
from moku.training import HF_MODEL

trainer.model.push_to_hub(HF_MODEL)
image_processor.push_to_hub(HF_MODEL)
print(f"Model pushed to https://huggingface.co/{HF_MODEL}")